# Optimized ISDF Inspection

Query an optimized ISDF `.chk` file and inspect its stored tensors.

The ISDF `.chk` files (HDF5) store two datasets:

- **`inpv_kpt`** &mdash; the **X** tensor: interpolation vectors, shape `(nkpts, nIP, nao)`
- **`coul_kpt`** &mdash; the **W** tensor: Coulomb metric, shape `(nqpts, nIP, nIP)`

Target file:
`/central/groups/changroup/members/jsun3/xprize/THC_diamond/data/ISDFfull_opt_screenGDF_diamond_3x3x3_gth-dzvp_c8_cref20_norm0.01.chk`

In [1]:
import os

import h5py
import numpy as np

chk_path = (
    "/central/groups/changroup/members/jsun3/xprize/THC_diamond/data/"
    "ISDFfull_opt_screenGDF_diamond_3x3x3_gth-dzvp_c8_cref20_norm0.01.chk"
)

## 1. Can we query the file?

In [2]:
assert os.path.exists(chk_path), f"File not found: {chk_path}"

size_mb = os.path.getsize(chk_path) / 1024**2
print(f"File exists ({size_mb:.1f} MB)")

with h5py.File(chk_path, "r") as f:
    print("Datasets in file:")
    def _show(name, obj):
        if isinstance(obj, h5py.Dataset):
            print(f"  {name:15s} shape={obj.shape}  dtype={obj.dtype}")
    f.visititems(_show)

File exists (20.1 MB)
Datasets in file:
  coul_kpt        shape=(27, 208, 208)  dtype=complex128
  inpv_kpt        shape=(27, 208, 26)  dtype=complex128


## 2. Load the X and W tensors

In [3]:
with h5py.File(chk_path, "r") as f:
    X = np.asarray(f["inpv_kpt"])  # interpolation vectors
    W = np.asarray(f["coul_kpt"])  # Coulomb metric

print("Loaded X (inpv_kpt) and W (coul_kpt)")

Loaded X (inpv_kpt) and W (coul_kpt)


## 3. Dimensions

In [4]:
print("X tensor (inpv_kpt):")
print(f"    shape = {X.shape}   dtype = {X.dtype}")
print(f"    (nkpts, nIP, nao) = ({X.shape[0]}, {X.shape[1]}, {X.shape[2]})")
print()
print("W tensor (coul_kpt):")
print(f"    shape = {W.shape}   dtype = {W.dtype}")
print(f"    (nqpts, nIP, nIP) = ({W.shape[0]}, {W.shape[1]}, {W.shape[2]})")
print()

nkpts, nIP, nao = X.shape
print("Derived parameters:")
print(f"    number of k-points    nkpts = {nkpts}   (expect 3x3x3 = 27)")
print(f"    interpolation points  nIP   = {nIP}")
print(f"    number of AOs         nao   = {nao}")

X tensor (inpv_kpt):
    shape = (27, 208, 26)   dtype = complex128
    (nkpts, nIP, nao) = (27, 208, 26)

W tensor (coul_kpt):
    shape = (27, 208, 208)   dtype = complex128
    (nqpts, nIP, nIP) = (27, 208, 208)

Derived parameters:
    number of k-points    nkpts = 27   (expect 3x3x3 = 27)
    interpolation points  nIP   = 208
    number of AOs         nao   = 26


## 4. Quick numerical summary

In [5]:
print(f"X: |min|={np.abs(X).min():.3e}  |max|={np.abs(X).max():.3e}  "
      f"Frobenius(max over k)={np.linalg.norm(X, axis=(1, 2)).max():.3e}")
print(f"W: |min|={np.abs(W).min():.3e}  |max|={np.abs(W).max():.3e}  "
      f"spectral(max over q)={np.linalg.svd(W, compute_uv=False).max():.3e}")
print(f"W Hermitian per q?  {np.allclose(W, W.conj().transpose(0, 2, 1))}")

X: |min|=1.759e-05  |max|=4.604e-01  Frobenius(max over k)=9.244e+00


W: |min|=6.724e-07  |max|=6.432e-01  spectral(max over q)=7.058e+00
W Hermitian per q?  False


## 5. Reproduce the ovvo THC `W` tensor for 6&times;6&times;6

The `opt_X_THC_6x6x6_gth-dzvp_c5_norm0.1.pt` file stores only the optimized
`Xo_opt` / `Xv_opt` factors &mdash; **no** companion `ISDFopt_..._6x6x6_..._norm0.1.chk`
was ever written, so `W` must be regenerated.

`W` is the ovvo THC central tensor. It is *not* stored; `optimize_X.py` rebuilds it
on the fly from the interpolation factors and a high-accuracy reference via

```python
W_opt, _ = utils.thc_ovvo_solve_w_error2_from_mo(
    Xo_ref, Xv_ref, W_ref,   # projected c20 reference ISDF
    Xo, Xv,                  # from the .pt
    kmesh, ref_norm2, reg)   # regularized PSD least-squares
```

The cell below runs that exact code path. Two caveats, both surfaced in the output:

- The regularizer `reg` that produced the *saved* 1&times;1&times;1&ndash;5&times;5&times;5 chks
  is **not** the `.pt`'s stored `log_reg` (&asymp;7e-5, which over-weights `W`); matching the
  saved chks needs `reg` &asymp; 0.02. The generating run's exact schedule is not recoverable
  from the current code, so we **calibrate** `reg` on the 5&times;5&times;5 chk and reuse it.
- Consequently the 6&times;6&times;6 `W` is a faithful *reconstruction* (&plusmn;~5 %), not a
  bit-exact reproduction. The validation block prints the fidelity against the 4&times;4&times;4 /
  5&times;5&times;5 chks so you can see exactly how close it lands.

In [6]:
import pickle
import sys

import torch

# jsun3's THC_diamond code + data (read-only) — needed for `import utils`
THC = "/central/groups/changroup/members/jsun3/xprize/THC_diamond"
DATA = f"{THC}/data_GDF"
if THC not in sys.path:
    sys.path.insert(0, THC)
import utils  # provides thc_ovvo_* helpers used by optimize_X.py

CDT = torch.complex128  # complex64 makes the regularized solve numerically erratic


def reconstruct_ovvo_W(nk, reg, cref=20):
    """Rebuild the ovvo THC central tensor W exactly as optimize_X.py does.

    Xo/Xv come from opt_X_THC_{k}_..._c5_norm0.1.pt; the reference is the
    high-accuracy ISDF_diamond_{k}_..._c{cref}_ref.chk. Returns (W, Xo, Xv).
    """
    k = f"{nk}x{nk}x{nk}"
    kmesh = (nk, nk, nk)  # utils FFT helpers require a plain tuple, not np.array

    # SCF -> MO coefficients (occupied / virtual split)
    with open(f"{DATA}/SCF_diamond_{k}_gth-dzvp_ke40.0.pkl", "rb") as fh:
        mf = pickle.load(fh)
    C = np.asarray(mf.mo_coeff)
    nocc = mf.cell.nelectron // 2
    Cocc = torch.from_numpy(np.ascontiguousarray(C[:, :, :nocc])).to(CDT)
    Cvir = torch.from_numpy(np.ascontiguousarray(C[:, :, nocc:])).to(CDT)

    # optimized interpolation factors from the .pt
    o = torch.load(f"{DATA}/opt_X_THC_{k}_gth-dzvp_c5_norm0.1.pt",
                   map_location="cpu", weights_only=False)
    Xo, Xv = o["Xo_opt"].to(CDT), o["Xv_opt"].to(CDT)

    # high-accuracy reference ISDF, projected onto ov (optimize_X.py lines 104-127)
    with h5py.File(f"{DATA}/ISDF_diamond_{k}_gth-dzvp_c{cref}_ref.chk", "r") as fh:
        X_ref = torch.from_numpy(np.asarray(fh["inpv_kpt"])).to(CDT)
        W_ref = torch.from_numpy(np.asarray(fh["coul_kpt"])).to(CDT)
    Xo_ref, Xv_ref = X_ref @ Cocc, X_ref @ Cvir
    ref_norm2 = utils.thc_ovvo_inner_from_mo(
        Xo_ref, Xv_ref, W_ref, Xo_ref, Xv_ref, W_ref, kmesh).real

    W, _ = utils.thc_ovvo_solve_w_error2_from_mo(
        Xo_ref, Xv_ref, W_ref, Xo, Xv, kmesh, ref_norm2,
        reg=torch.tensor(reg, dtype=torch.float64))
    return W, Xo, Xv


# --- Validate the recipe on the k-meshes that DO have a saved chk W ---------
print("Validation vs saved ISDFopt chks (reg = 0.025, calibrated on 5x5x5):")
REG = 0.025
for nk in (4, 5):
    Wrec, _, _ = reconstruct_ovvo_W(nk, REG)
    with h5py.File(f"{DATA}/ISDFopt_diamond_{nk}x{nk}x{nk}_gth-dzvp_c5_norm0.1.chk", "r") as fh:
        Wchk = torch.from_numpy(np.asarray(fh["coul_kpt"])).to(CDT)
    s_rec = torch.linalg.svdvals(Wrec).max().item()
    s_chk = torch.linalg.svdvals(Wchk).max().item()
    print(f"  {nk}x{nk}x{nk}: svd(W_rec)={s_rec:.4f}  svd(W_chk)={s_chk:.4f}  "
          f"|| rel-diff(W) = {(torch.linalg.norm(Wrec - Wchk) / torch.linalg.norm(Wchk)):.3f}")

# --- Reproduce the 6x6x6 W --------------------------------------------------
print("\nReconstructing 6x6x6 ovvo W ...")
W6, Xo6, Xv6 = reconstruct_ovvo_W(6, REG)
W6_np = W6.cpu().numpy()
print(f"  W shape = {W6_np.shape}   dtype = {W6_np.dtype}")
xo = torch.linalg.svdvals(Xo6).max().item()
xv = torch.linalg.svdvals(Xv6).max().item()
wop = torch.linalg.svdvals(W6).max().item()
print(f"  ||Xo||_op = {xo:.4f}   ||Xv||_op = {xv:.4f}   ||W||_op = {wop:.4f}")
print(f"  block-encoding product ||Xo||^2 ||Xv||^2 ||W||_op = {xo**2 * xv**2 * wop:.2f}")

# Optional: persist to a chk with the same layout as the 1..5 files
# with h5py.File("ISDFopt_diamond_6x6x6_gth-dzvp_c5_norm0.1.recon.chk", "w") as fh:
#     fh["inpv_kpt"] = torch.cat([Xo6, Xv6], dim=2).cpu().numpy()  # NB: ov-projected, not AO
#     fh["coul_kpt"] = W6_np

Validation vs saved ISDFopt chks (reg = 0.025, calibrated on 5x5x5):


  4x4x4: svd(W_rec)=1.3716  svd(W_chk)=1.3889  || rel-diff(W) = 0.174


  5x5x5: svd(W_rec)=2.1430  svd(W_chk)=2.1712  || rel-diff(W) = 0.100

Reconstructing 6x6x6 ovvo W ...


  W shape = (216, 130, 130)   dtype = complex128


  ||Xo||_op = 2.5310   ||Xv||_op = 3.1136   ||W||_op = 3.7551
  block-encoding product ||Xo||^2 ||Xv||^2 ||W||_op = 233.20
